# Python compile() Function Interactive Practice

Welcome to the interactive companion notebook for the **[plus2net Python compile() Tutorial](https://www.plus2net.com/python/compile.php)**.

In this workbook, you will practice:
1. Compiling source code across `'exec'`, `'eval'`, and `'single'` modes.
2. Benchmarking execution speedup from precompiled bytecode in loops.
3. Compiling and executing Abstract Syntax Tree (`ast.AST`) objects.
4. Inspecting code object metadata and disassembling instructions using `dis.dis()`.
5. Implementing a secure, AST-validated safe expression evaluator.
6. Running self-verifying unit assertions.

### 1. Basic Compilation Modes: 'exec' vs 'eval' vs 'single'

In [ ]:
# 1. 'exec' mode for multi-statement sequences
exec_code = compile("""
numbers = [1, 2, 3, 4, 5]
squared = [n ** 2 for n in numbers]
print('Squared list:', squared)
""", filename="<demo_exec>", mode="exec")

exec(exec_code)

# 2. 'eval' mode for single expressions returning values
eval_code = compile("sum([10, 20, 30]) * 2", filename="<demo_eval>", mode="eval")
total = eval(eval_code)
print("Evaluated total:", total)

# 3. 'single' mode for interactive REPL statements
single_code = compile("'python'.capitalize()", filename="<demo_single>", mode="single")
exec(single_code)


### 2. Performance Benchmark: Precompiled Bytecode in Loops
Compare repeatedly evaluating a raw string vs. precompiling the expression once.

In [ ]:
import time

expression = "x * 2 + 10"
iterations = 50000

# Raw string eval
t0 = time.perf_counter()
for x in range(iterations):
    _ = eval(expression)
raw_time = time.perf_counter() - t0

# Precompiled bytecode eval
compiled_obj = compile(expression, filename="<bench>", mode="eval")
t1 = time.perf_counter()
for x in range(iterations):
    _ = eval(compiled_obj)
precompiled_time = time.perf_counter() - t1

print(f"Raw string eval:     {raw_time:.4f} s")
print(f"Precompiled bytecode:{precompiled_time:.4f} s")
print(f"Speedup:             {raw_time / precompiled_time:.2f}x faster")


### 3. Compiling Abstract Syntax Tree (AST) Objects
Parse source code into an AST node tree, inspect the nodes, and compile the AST into bytecode.

In [ ]:
import ast

source = "result = [x * 3 for x in range(5)]"
tree = ast.parse(source, mode="exec")

print("Root AST node:", ast.dump(tree)[:120], "...")

# Compile the AST tree
code_obj = compile(tree, filename="<ast_tree>", mode="exec")
local_ns = {}
exec(code_obj, local_ns)
print("Computed result in namespace:", local_ns["result"])


### 4. Code Object Introspection & Disassembly with dis.dis()
Examine bytecode instructions generated by Python's compiler.

In [ ]:
import dis

calc_code = compile("a * b + 15", filename="<dis_demo>", mode="eval")

print("Constants (co_consts): ", calc_code.co_consts)
print("Referenced names:      ", calc_code.co_names)
print("\nDisassembled bytecode:")
dis.dis(calc_code)


### 5. AST-Validated Safe Expression Evaluator
Safely evaluate user math expressions by rejecting function calls, attribute access, and imports.

In [ ]:
import ast
def safe_eval(expr_str: str, variables: dict) -> float:
    tree = ast.parse(expr_str, mode='eval')
    for node in ast.walk(tree):
        if not isinstance(node, (ast.Expression, ast.BinOp, ast.UnaryOp, 
                                ast.Constant, ast.Name, ast.Load,
                                ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow)):
            raise ValueError(f"Disallowed element: {type(node).__name__}")
    compiled = compile(tree, filename="<safe>", mode="eval")
    return eval(compiled, {"__builtins__": {}}, variables)

# Test safe expression
val = safe_eval("base * rate + bonus", {"base": 100, "rate": 1.25, "bonus": 50})
print("Safe calculation:", val)

# Test unsafe expression
try:
    safe_eval("__import__('os').getcwd()", {})
except ValueError as err:
    print("Safely blocked:", err)


### 6. Self-Checking Unit Verification Suite
Run all test assertions to verify compilation mechanics.

In [ ]:
# Verify exec mode
c_exec = compile("val = 40 + 2", "<test>", "exec")
ns = {}
exec(c_exec, ns)
assert ns["val"] == 42

# Verify eval mode
c_eval = compile("10 * 5", "<test>", "eval")
assert eval(c_eval) == 50

# Verify optimization level 2 strips docstrings
c_opt = compile("'''demo'''\nx = 1", "<opt>", "exec", optimize=2)
assert "demo" not in c_opt.co_consts

print("All compile() unit assertions successfully verified! 100% passed.")
